In [4]:
import torch
from torch.autograd import gradcheck

def my_function(x):
    return (x**3).sum()

# gradcheck requires double precision
x = torch.randn(5, dtype=torch.double, requires_grad=True)

test = gradcheck(my_function, (x,), eps=1e-6, atol=1e-4)
print("Grad correct:", test)


Grad correct: True


In [5]:
import torch

def grad_check(f, x, eps=1e-6, tol=1e-4):
    """
    f : function that returns scalar
    x : tensor with requires_grad=True
    """

    # -------- Analytical gradient --------
    x = x.clone().detach().requires_grad_(True)
    y = f(x)
    y.backward()
    grad_autograd = x.grad.clone()

    # -------- Numerical gradient --------
    grad_numeric = torch.zeros_like(x)

    for i in range(x.numel()):
        x_pos = x.clone().detach()
        x_neg = x.clone().detach()

        x_pos.view(-1)[i] += eps
        x_neg.view(-1)[i] -= eps

        y_pos = f(x_pos)
        y_neg = f(x_neg)

        grad_numeric.view(-1)[i] = (y_pos - y_neg) / (2 * eps)

    # -------- Compare --------
    diff = torch.norm(grad_autograd - grad_numeric)
    norm = torch.norm(grad_autograd) + torch.norm(grad_numeric)
    rel_error = diff / norm

    print("Autograd:", grad_autograd)
    print("Numeric :", grad_numeric)
    print("Relative error:", rel_error.item())

    return rel_error < tol


In [11]:
def f(x):
    return (x**2).sum()

x = torch.randn(2, requires_grad=True)

print(grad_check(f, x))


Autograd: tensor([1.9638, 2.5145])
Numeric : tensor([2.0266, 2.3842])
Relative error: 0.022889824584126472
tensor(False)
